# Best-3 Torus Model Selection: Velocity Disentanglement, Test-Time Disease, and Future Shape

This notebook compares only the three selected models:

- `Subset 64/128/64 best`, checkpoint 500
- `Subset 64/160/32 E2 best saved`, checkpoint 1000
- `Additive strong best`, checkpoint 1000

It does **not** display heavy tables or large 3D figures inline. Every output is saved into an indexed HTML report:

`analysis_torus_best3_disentanglement_future_speed/figures/index.html`

The goal is to decide:

- which model has the cleanest velocity disentanglement;
- which model supports label-free test-time disease prediction;
- which model is best for future generation;
- how latent speed, shape-space speed, INR differences, and decoder Jacobian maps should be interpreted.


In [1]:
# Configuration, imports, and notebook-local HTML report helpers.
import csv
import html as html_lib
import json
import math
import random
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
import trimesh
from IPython.display import HTML, display
from plotly.subplots import make_subplots
from skimage.measure import marching_cubes

ROOT = Path('/home/jakaria/INR/Deep3DComp')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from deep_sdf import data as deep_sdf_data
from networks.deep_sdf_decoder import Decoder
from networks.longitudinal_disentangled_flow_64_128_64_adv import (
    build_temporal_flow as build_subset128_flow,
)
from networks.longitudinal_flow_64_160_32_pred_dx import (
    build_temporal_flow as build_subset160_flow,
)
from networks.longitudinal_additive_flow import (
    build_temporal_flow as build_additive_flow,
)

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

GPU_ID = 0
if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)
DEVICE = torch.device(f'cuda:{GPU_ID}' if torch.cuda.is_available() else 'cpu')

# Surface settings. Reduce GRID_RES to 48 if GPU memory is tight.
AGES = (60.0, 70.0, 80.0)
REFERENCE_AGE = 60.0
GRID_RES = 64
GRID_BATCH = 2**17
SURFACE_BATCH = 2048
DELTA_T = 0.10
FD_EPS = 1e-3
EVAL_DIAGNOSIS = 1.0
COMPONENTS = ('age', 'disease_raw', 'disease', 'residual', 'total')

OUTPUT_DIR = ROOT / 'analysis_torus_best3_disentanglement_future_speed'
FIGURE_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SAVE_INTERACTIVE_HTML = True
SHOW_INLINE_PLOTLY = False
SHOW_INLINE_MATPLOTLIB = False
FIGURE_MANIFEST = {}

MODEL_COLORS = {
    'Subset 64/128/64 best': '#C44536',
    'Subset 64/160/32 E2 best saved': '#147D92',
    'Additive strong best': '#D97706',
}
SHORT_NAMES = {
    'Subset 64/128/64 best': '64/128/64',
    'Subset 64/160/32 E2 best saved': '64/160/32',
    'Additive strong best': 'additive',
}


def figure_slug(value):
    text = re.sub(r'[^A-Za-z0-9._-]+', '_', str(value)).strip('_')
    return text or 'figure'


def register_output(path, title, category, description=''):
    path = Path(path)
    FIGURE_MANIFEST[str(path)] = {
        'path': path,
        'title': str(title),
        'category': str(category),
        'description': str(description),
    }
    print('Saved output:', path)


def html_page(title, body, description=''):
    desc = f"<p class='note'>{html_lib.escape(str(description))}</p>" if description else ''
    return f"""<!doctype html>
<html><head><meta charset='utf-8'><title>{html_lib.escape(str(title))}</title>
<style>
body{{font-family:Arial,sans-serif;max-width:1200px;margin:32px auto;padding:0 22px;color:#202020}}
h1{{margin-bottom:8px}} h2{{margin-top:28px}} p{{line-height:1.55}}
.note{{background:#edf6f5;border-left:4px solid #147d92;padding:12px 14px;border-radius:6px}}
.callout{{background:#fff7ed;border-left:4px solid #d97706;padding:12px 14px;border-radius:6px}}
code{{background:#f3f3f3;padding:2px 5px;border-radius:3px}}
a{{color:#0b5cad;text-decoration:none}} a:hover{{text-decoration:underline}}
img{{max-width:100%;border:1px solid #ddd;border-radius:8px}}
</style></head><body><h1>{html_lib.escape(str(title))}</h1>{desc}{body}</body></html>"""


def save_text_page(filename, title, category, body_html, description=''):
    html_path = FIGURE_DIR / f'{figure_slug(filename)}.html'
    html_path.write_text(html_page(title, body_html, description), encoding='utf-8')
    register_output(html_path, title, category, description)
    return html_path


def save_plotly_figure(fig, filename, title, category, description='', show_inline=False):
    html_path = FIGURE_DIR / f'{figure_slug(filename)}.html'
    if SAVE_INTERACTIVE_HTML:
        fig.write_html(html_path, include_plotlyjs='directory', full_html=True, auto_open=False)
        register_output(html_path, title, category, description)
    if SHOW_INLINE_PLOTLY and show_inline:
        fig.show()
    return html_path


def save_matplotlib_figure(fig, filename, title, category, description='', dpi=170):
    stem = figure_slug(filename)
    png_path = FIGURE_DIR / f'{stem}.png'
    html_path = FIGURE_DIR / f'{stem}.html'
    fig.savefig(png_path, dpi=dpi, bbox_inches='tight', facecolor='white')
    body = f"<p><img src='{png_path.name}' alt='{html_lib.escape(str(title))}'></p>"
    html_path.write_text(html_page(title, body, description), encoding='utf-8')
    register_output(html_path, title, category, description)
    if SHOW_INLINE_MATPLOTLIB:
        plt.show()
    else:
        plt.close(fig)
    return html_path


def write_figure_index(report_title='Best-3 Torus Model Selection Report'):
    entries = sorted(
        FIGURE_MANIFEST.values(),
        key=lambda item: (item['category'], item['title'], str(item['path'])),
    )
    groups = {}
    for item in entries:
        groups.setdefault(item['category'], []).append(item)
    chunks = [
        "<!doctype html><html><head><meta charset='utf-8'>",
        f"<title>{html_lib.escape(report_title)}</title>",
        "<style>body{font-family:Arial,sans-serif;max-width:1150px;margin:32px auto;padding:0 20px;color:#202020}",
        "h1{margin-bottom:8px}h2{margin-top:30px}li{margin:10px 0}a{color:#0b5cad;text-decoration:none}",
        "a:hover{text-decoration:underline}.desc{color:#555;font-size:14px}</style></head><body>",
        f"<h1>{html_lib.escape(report_title)}</h1>",
        "<p>This report contains plots and explanation pages only. No large notebook-inline figures are required.</p>",
        f"<p><strong>Output directory:</strong> <code>{html_lib.escape(str(FIGURE_DIR))}</code></p>",
    ]
    for category, items in groups.items():
        chunks.append(f"<h2>{html_lib.escape(category)}</h2><ul>")
        for item in items:
            relative = item['path'].relative_to(FIGURE_DIR)
            desc = html_lib.escape(item.get('description', ''))
            chunks.append(
                f"<li><a href='{relative.as_posix()}'>{html_lib.escape(item['title'])}</a>"
                f"<br><span class='desc'>{desc}</span></li>"
            )
        chunks.append('</ul>')
    chunks.append('</body></html>')
    index_path = FIGURE_DIR / 'index.html'
    index_path.write_text(''.join(chunks), encoding='utf-8')
    print('Figure index:', index_path)
    return index_path

print('Device:', DEVICE)
print('Output folder:', FIGURE_DIR)


Device: cuda:0
Output folder: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures


## How to read the colors and speeds

The surface plots use signed normal speed:

\[
 c_k(x,t)=-\frac{\nabla_z F(z_t,x)^\top v_k}{\|\nabla_xF(z_t,x)\|+\epsilon}.
\]

Interpretation:

- **Red**: the component pushes the zero-level surface outward.
- **Blue**: the component pulls the surface inward.
- **White/neutral**: little local shape change.
- **Latent speed** \(\|v\|\) is not anatomical by itself.
- **Shape speed** is the decoder-induced surface motion and is more important for anatomical interpretation.
- **INR difference** uses the professor's requested finite-step view: `INR(z) - INR(z + v * t)`.
- **Jacobian sensitivity** shows where the decoder is sensitive to latent changes, independent of a specific disease direction.


In [2]:
# Load existing CSV outputs from Part 1 and Part 2.
P1 = ROOT / 'analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_1' / 'figures'
P2 = ROOT / 'analysis_torus_shape_space_velocity_subset_64_128_vs_64_160_vs_additive_best_part_2' / 'figures'

CSV_PATHS = {
    'checkpoint': P1 / '01_checkpoint_selection.csv',
    'diagnosis': P1 / '02_64_160_disease_prediction_table.csv',
    'real_summary': P1 / '04_real_speed_summary_table.csv',
    'mean_metrics': P1 / '06_mean_anchor_speed_metrics.csv',
    'mean_summary': P1 / '07_mean_anchor_speed_summary.csv',
    'heldout_speed': P2 / '02_heldout_component_speed_table.csv',
    'heldout_real': P2 / '05_heldout_real_speed_table.csv',
    'heldout_summary': P2 / '06_heldout_model_speed_summary.csv',
    'future': P2 / '08_64_160_future_prediction_table.csv',
    'future_summary': P2 / '09_64_160_future_prediction_summary.csv',
    'jacobian': P2 / '11_jacobian_sensitivity_table.csv',
}

missing = [name for name, path in CSV_PATHS.items() if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing previous result CSVs: {missing}')

CHECKPOINT = pd.read_csv(CSV_PATHS['checkpoint'])
DIAGNOSIS_METRICS = pd.read_csv(CSV_PATHS['diagnosis'])
REAL_SUMMARY = pd.read_csv(CSV_PATHS['real_summary'])
MEAN_METRICS = pd.read_csv(CSV_PATHS['mean_metrics'])
MEAN_SUMMARY = pd.read_csv(CSV_PATHS['mean_summary'])
HELDOUT_SPEED = pd.read_csv(CSV_PATHS['heldout_speed'])
HELDOUT_REAL = pd.read_csv(CSV_PATHS['heldout_real'])
HELDOUT_SUMMARY = pd.read_csv(CSV_PATHS['heldout_summary'])
FUTURE_64_160 = pd.read_csv(CSV_PATHS['future'])
FUTURE_SUMMARY = pd.read_csv(CSV_PATHS['future_summary'])
JACOBIAN_SENSITIVITY = pd.read_csv(CSV_PATHS['jacobian'])

save_text_page(
    '00_reading_guide',
    'Reading guide: what each plot means',
    'Explanation',
    """
    <p>This notebook separates three questions that are often mixed together:</p>
    <h2>1. Velocity disentanglement</h2>
    <p>A good disentangled model puts disease-related shape change in the disease component, ordinary aging in the age component, and little coherent progression in residual.</p>
    <h2>2. Test-time disease prediction</h2>
    <p>Only the <code>64/160/32</code> model has a baseline-anchor disease head. The old <code>64/128/64</code> and additive models use an oracle disease condition in these visualizations.</p>
    <h2>3. Future generation</h2>
    <p>Forecast quality must be judged separately from velocity disentanglement. A model may reconstruct future scans well while using a poorly separated velocity decomposition.</p>
    <p class='callout'><strong>Important:</strong> the current future volume-error numbers have a mesh-scale mismatch. Use SDF MAE and visual shape comparison first; do not rank models by the current volume error.</p>
    """,
    'Definitions for interpreting the saved plots and final model choice.',
)

print('Loaded previous CSV outputs.')


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/00_reading_guide.html
Loaded previous CSV outputs.


In [3]:
# Existing-result plots: disease prediction, reported forecast quality, residual leakage, and held-out speed behavior.
def add_model_color(fig):
    for trace in fig.data:
        name = getattr(trace, 'name', None)
        if name in MODEL_COLORS:
            trace.marker.color = MODEL_COLORS[name]
            if hasattr(trace, 'line'):
                trace.line.color = MODEL_COLORS[name]
    return fig

# Disease prediction for 64/160/32.
diag = DIAGNOSIS_METRICS.copy().sort_values('epoch')
fig = make_subplots(rows=1, cols=2, subplot_titles=('Disease prediction', 'Label-free forecast CSV'))
fig.add_trace(go.Scatter(x=diag['epoch'], y=diag['balanced_accuracy'], mode='lines+markers', name='balanced accuracy'), row=1, col=1)
fig.add_trace(go.Scatter(x=diag['epoch'], y=diag['auc'], mode='lines+markers', name='AUC'), row=1, col=1)
fig.add_trace(go.Scatter(x=diag['epoch'], y=diag['forecast_mean_chamfer'], mode='lines+markers', name='mean future Chamfer'), row=1, col=2)
fig.add_trace(go.Scatter(x=diag['epoch'], y=diag['forecast_final_chamfer'], mode='lines+markers', name='final future Chamfer'), row=1, col=2)
selected = diag[diag['selected_for_notebook'].astype(bool)]
if len(selected):
    selected_epoch = float(selected['epoch'].iloc[0])
    fig.add_vline(x=selected_epoch, line_dash='dash', line_color='black', row=1, col=1)
    fig.add_vline(x=selected_epoch, line_dash='dash', line_color='black', row=1, col=2)
fig.update_layout(
    title='64/160/32 test-time disease prediction and future CSV metrics',
    width=1250,
    height=470,
)
fig.update_yaxes(range=[0, 1.05], row=1, col=1)
save_plotly_figure(
    fig,
    '01_64_160_disease_prediction_and_forecast_curves',
    '64/160/32 disease prediction and forecast curves',
    'Prediction and generation',
    'Only 64/160/32 has label-free disease prediction. The dashed line is the loaded checkpoint.',
)

# Reported test Chamfer for all three loaded checkpoints.
ck = CHECKPOINT.copy().sort_values('reported_or_current_test_chamfer')
fig = go.Figure()
fig.add_trace(go.Bar(
    x=ck['model'],
    y=ck['reported_or_current_test_chamfer'],
    marker_color=[MODEL_COLORS.get(m, '#888') for m in ck['model']],
    text=[f"{v:.2e}" for v in ck['reported_or_current_test_chamfer']],
    textposition='outside',
))
fig.update_layout(
    title='Reported/saved checkpoint future Chamfer comparison',
    xaxis_title='model',
    yaxis_title='Chamfer, lower is better',
    width=1000,
    height=520,
    margin=dict(b=130),
)
fig.update_xaxes(tickangle=20)
save_plotly_figure(
    fig,
    '02_reported_checkpoint_chamfer',
    'Reported checkpoint Chamfer comparison',
    'Prediction and generation',
    'This compares saved checkpoint forecast quality, not disentanglement.',
)

# Disentanglement summary from mean-anchor and held-out speed reports.
held = HELDOUT_SUMMARY.copy()
held_model = held.groupby('model').agg(
    mean_age=('age', 'mean'),
    mean_disease=('disease', 'mean'),
    mean_residual=('residual', 'mean'),
    mean_total=('total', 'mean'),
    mean_residual_fraction=('residual_fraction', 'mean'),
).reset_index()
held_model['disease_to_residual'] = held_model['mean_disease'] / (held_model['mean_residual'] + 1e-12)
healthy = held[held['true_diagnosis'] == 0][['model', 'disease', 'residual', 'total', 'gate_used']].rename(columns={'disease': 'healthy_false_disease_speed'})
diseased = held[held['true_diagnosis'] == 1][['model', 'disease', 'residual', 'total', 'gate_used']].rename(columns={'disease': 'diseased_disease_speed'})
held_model = held_model.merge(healthy[['model', 'healthy_false_disease_speed']], on='model', how='left')
held_model = held_model.merge(diseased[['model', 'diseased_disease_speed']], on='model', how='left')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Residual fraction, lower is better',
        'Disease / residual speed, higher is better',
        'Healthy false disease speed, lower is better',
        'Diseased disease speed, higher is better',
    ),
)
for col, metric in enumerate(['mean_residual_fraction', 'disease_to_residual'], start=1):
    fig.add_trace(go.Bar(
        x=held_model['model'], y=held_model[metric],
        marker_color=[MODEL_COLORS.get(m, '#888') for m in held_model['model']],
        showlegend=False,
    ), row=1, col=col)
for col, metric in enumerate(['healthy_false_disease_speed', 'diseased_disease_speed'], start=1):
    fig.add_trace(go.Bar(
        x=held_model['model'], y=held_model[metric],
        marker_color=[MODEL_COLORS.get(m, '#888') for m in held_model['model']],
        showlegend=False,
    ), row=2, col=col)
fig.update_layout(
    title='Held-out velocity disentanglement summary',
    width=1250,
    height=760,
    margin=dict(b=150),
)
fig.update_xaxes(tickangle=22)
save_plotly_figure(
    fig,
    '03_heldout_disentanglement_summary',
    'Held-out velocity disentanglement summary',
    'Velocity disentanglement',
    'A good model has low residual fraction, high diseased disease speed, and low healthy false disease speed.',
)

# Jacobian sensitivity from previous report.
fig = go.Figure()
for model, sub in JACOBIAN_SENSITIVITY.groupby('model'):
    fig.add_trace(go.Scatter(
        x=sub['sid'].astype(str),
        y=sub['mean_latent_jacobian_norm'],
        mode='lines+markers',
        name=model,
        line=dict(color=MODEL_COLORS.get(model, None)),
    ))
fig.update_layout(
    title='Decoder latent-Jacobian sensitivity on held-out surfaces',
    xaxis_title='test subject',
    yaxis_title='mean ||d INR / dz||',
    width=1000,
    height=470,
)
save_plotly_figure(
    fig,
    '04_decoder_jacobian_sensitivity_from_previous_report',
    'Decoder latent-Jacobian sensitivity from previous report',
    'Jacobian sensitivity',
    'Large latent velocity only matters if the decoder Jacobian converts it into shape change.',
)


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/01_64_160_disease_prediction_and_forecast_curves.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/02_reported_checkpoint_chamfer.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/03_heldout_disentanglement_summary.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/04_decoder_jacobian_sensitivity_from_previous_report.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/04_decoder_jacobian_sensitivity_from_previous_report.html')

In [4]:
# Ground-truth real speed plots from metadata: torus thickness speed and disease bump speed.
def load_label_dataframe(labels_path):
    obj = torch.load(labels_path, map_location='cpu')
    rows = []
    for scan_id, payload in obj.items():
        row = {'scan_id': str(scan_id)}
        row.update(payload)
        rows.append(row)
    df = pd.DataFrame(rows)
    df['sid'] = df['scan_id'].str.extract(r'ID_(\d+)_t').astype(int)
    df['tp'] = df['scan_id'].str.extract(r'_t(\d+)').astype(int)
    return df.sort_values(['sid', 'tp']).reset_index(drop=True)


def observed_rate_table(label_df):
    rows = []
    for sid, group in label_df.groupby('sid'):
        group = group.sort_values('tp')
        for left, right in zip(group.iloc[:-1].to_dict('records'), group.iloc[1:].to_dict('records')):
            dt_age = float(right['age']) - float(left['age'])
            dt_norm = float(right['age_norm']) - float(left['age_norm'])
            if abs(dt_age) < 1e-8 or abs(dt_norm) < 1e-8:
                continue
            rows.append({
                'sid': sid,
                'diagnosis': int(left['diagnosis']),
                'from_scan': left['scan_id'],
                'to_scan': right['scan_id'],
                'age_mid': 0.5 * (float(left['age']) + float(right['age'])),
                'age_norm_mid': 0.5 * (float(left['age_norm']) + float(right['age_norm'])),
                'dt_years': dt_age,
                'dt_norm': dt_norm,
                'thickness_rate_per_year': (float(right['thickness']) - float(left['thickness'])) / dt_age,
                'bump_rate_per_year': (float(right['bump_height']) - float(left['bump_height'])) / dt_age,
                'thickness_rate_per_norm_time': (float(right['thickness']) - float(left['thickness'])) / dt_norm,
                'bump_rate_per_norm_time': (float(right['bump_height']) - float(left['bump_height'])) / dt_norm,
            })
    return pd.DataFrame(rows)

# Use the metadata path already recorded in the checkpoint specs.
LABELS_PATH = ROOT / 'dummy'
for _, row in CHECKPOINT.iterrows():
    specs_path = Path(row['model_key'])
# Direct path from the experiment specs is loaded later; this fixed path is shared by all three torus runs.
LABELS_PATH = Path('/home/jakaria/torus_creation/torus_age_disease_longitudinal_100ids_5tp_flat/labels.pt')
LABEL_DF = load_label_dataframe(LABELS_PATH)
REAL_RATE_DF = observed_rate_table(LABEL_DF)

fig = make_subplots(rows=1, cols=2, subplot_titles=('Torus thickness speed', 'Disease bump speed'))
for diag, label, color in [(0, 'healthy', '#147D92'), (1, 'diseased', '#D1495B')]:
    sub = REAL_RATE_DF[REAL_RATE_DF['diagnosis'] == diag]
    fig.add_trace(go.Scatter(
        x=sub['age_mid'], y=sub['thickness_rate_per_norm_time'],
        mode='markers', name=f'{label} thickness',
        marker=dict(color=color, size=6, opacity=0.55),
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=sub['age_mid'], y=sub['bump_rate_per_norm_time'],
        mode='markers', name=f'{label} bump',
        marker=dict(color=color, size=6, opacity=0.55),
    ), row=1, col=2)
for col in (1, 2):
    fig.add_hline(y=0.0, line_color='black', line_width=1, row=1, col=col)
fig.update_layout(
    title='Ground-truth finite-difference speeds from real torus metadata',
    width=1250,
    height=500,
)
fig.update_xaxes(title='age midpoint')
fig.update_yaxes(title='rate per normalized time')
save_plotly_figure(
    fig,
    '05_ground_truth_torus_and_bump_speed_scatter',
    'Ground-truth torus thickness and bump speed',
    'Ground-truth speed',
    'These are real finite differences from metadata, not model outputs. Thickness is ordinary torus aging; bump speed is disease progression.',
)

# Bin real speeds near 60,70,80 for comparison with model age-specific speeds.
REAL_RATE_DF['nearest_age'] = REAL_RATE_DF['age_mid'].apply(lambda x: min(AGES, key=lambda a: abs(a - x)))
REAL_AGE_SUMMARY = (
    REAL_RATE_DF.groupby(['nearest_age', 'diagnosis'])
    .agg(
        thickness_rate=('thickness_rate_per_norm_time', 'mean'),
        bump_rate=('bump_rate_per_norm_time', 'mean'),
        n=('sid', 'size'),
    )
    .reset_index()
)
REAL_AGE_SUMMARY.to_csv(OUTPUT_DIR / 'real_age_speed_summary.csv', index=False)
print('Saved real age speed summary CSV for reproducibility, but report uses plots only.')


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/05_ground_truth_torus_and_bump_speed_scatter.html
Saved real age speed summary CSV for reproducibility, but report uses plots only.


## Load the three models for AUC and age-specific shape-space speed

The next cells load the actual checkpoints and compute new plots. This is needed for:

- train-subject component-speed AUC;
- latent speed versus shape-space speed;
- disease surface speed maps at ages 60, 70, and 80;
- INR finite-step maps;
- decoder Jacobian sensitivity maps.

These cells may take longer than the CSV-only summary cells because they decode implicit surfaces and evaluate decoder derivatives.


In [5]:
# Load the three selected models and their trained subject anchors.
BASE = ROOT / 'examples' / 'Torus_subset_100_id_age_progression'
SUBSET128_DIR = BASE / 'longitudinal_age_disease_conditioned_cocycle_shape_pair_loss_multiple_pairs_disentangled_velocity_64_128_64_binary_margin_adv'
SUBSET160_DIR = BASE / 'velocity_64_160_32_baseline_dx'
ADDITIVE_DIR = BASE / 'longitudinal_age_disease_additive_velocity_disentanglement_strong_residual_invariance_adv'

MODEL_CONFIGS = {
    'subset64_128_best': {
        'label': 'Subset 64/128/64 best',
        'short_label': '64/128/64',
        'kind': 'subset128',
        'exp_dir': SUBSET128_DIR,
        'checkpoint': '500',
    },
    'subset64_160_best_saved': {
        'label': 'Subset 64/160/32 E2 best saved',
        'short_label': '64/160/32',
        'kind': 'subset160',
        'exp_dir': SUBSET160_DIR,
        'checkpoint': '1000',
    },
    'additive_best': {
        'label': 'Additive strong best',
        'short_label': 'additive',
        'kind': 'additive',
        'exp_dir': ADDITIVE_DIR,
        'checkpoint': '1000',
    },
}


def checkpoint_path(exp_dir, subdir, checkpoint):
    name = str(checkpoint)
    if not name.endswith('.pth'):
        name += '.pth'
    path = Path(exp_dir) / subdir / name
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def strip_module_prefix(state):
    return {k.removeprefix('module.'): v for k, v in state.items()}


def latent_weights(payload):
    lat = payload.get('latent_codes_state_dict', payload.get('latent_codes'))
    if isinstance(lat, dict):
        lat = lat['weight']
    if lat.ndim == 3 and lat.shape[1] == 1:
        lat = lat[:, 0]
    return lat.detach().float()


def flow_builder(kind):
    if kind == 'subset128':
        return build_subset128_flow
    if kind == 'subset160':
        return build_subset160_flow
    if kind == 'additive':
        return build_additive_flow
    raise ValueError(kind)


def load_model(config):
    exp_dir = Path(config['exp_dir'])
    specs = json.loads((exp_dir / 'specs.json').read_text())
    decoder = Decoder(int(specs['CodeLength']), **specs['NetworkSpecs']).to(DEVICE)
    flow = flow_builder(config['kind'])(
        specs,
        int(specs['CodeLength']),
        list(specs.get('FlowHiddenDims', [256, 256])),
        age_condition_dim=int(specs.get('AgeConditionDim', 0)),
    ).to(DEVICE)
    model_payload = torch.load(checkpoint_path(exp_dir, 'ModelParameters', config['checkpoint']), map_location=DEVICE)
    decoder.load_state_dict(strip_module_prefix(model_payload['model_state_dict']))
    flow.load_state_dict(strip_module_prefix(model_payload['flow_state_dict']))
    decoder.eval(); flow.eval()
    latent_payload = torch.load(checkpoint_path(exp_dir, 'LatentCodes', config['checkpoint']), map_location='cpu')
    weights = latent_weights(latent_payload)
    return {
        **config,
        'specs': specs,
        'decoder': decoder,
        'flow': flow,
        'latent_weights': weights,
        'mean_z': weights.mean(dim=0, keepdim=True).to(DEVICE),
        'epoch': int(model_payload.get('epoch', -1)),
    }

MODELS = {name: load_model(cfg) for name, cfg in MODEL_CONFIGS.items()}
print('Loaded models:', [m['label'] for m in MODELS.values()])


Loaded models: ['Subset 64/128/64 best', 'Subset 64/160/32 E2 best saved', 'Additive strong best']


In [6]:
# Component extraction, AUC helpers, and train-subject latent-speed probe.
def auc_score(y, score):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    pos = np.flatnonzero(y == 1)
    neg = np.flatnonzero(y == 0)
    if len(pos) == 0 or len(neg) == 0:
        return np.nan
    order = np.argsort(score)
    ranks = np.empty(len(score), dtype=float)
    i = 0
    while i < len(score):
        j = i + 1
        while j < len(score) and score[order[j]] == score[order[i]]:
            j += 1
        ranks[order[i:j]] = 0.5 * (i + 1 + j)
        i = j
    return float((ranks[pos].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))


def zero_pad_subset(model, comp, block):
    flow = model['flow']
    z = comp.new_zeros(comp.shape[0], flow.age_dim + flow.disease_dim + flow.residual_dim)
    if block == 'age':
        z[:, :flow.age_dim] = comp
    elif block == 'disease':
        z[:, flow.age_dim:flow.age_dim + flow.disease_dim] = comp
    elif block == 'residual':
        z[:, flow.age_dim + flow.disease_dim:] = comp
    else:
        raise ValueError(block)
    return z


def component_vectors(model, z, time_value=0.5, diagnosis=1.0):
    flow = model['flow']
    t = torch.full((z.shape[0], 1), float(time_value), device=z.device, dtype=z.dtype)
    cond = torch.full((z.shape[0], 1), float(diagnosis), device=z.device, dtype=z.dtype)
    with torch.no_grad():
        comp = flow.velocity_components(z, t, t, age_cond=cond)
    if model['kind'] in ('subset128', 'subset160'):
        age = zero_pad_subset(model, comp['age'], 'age')
        disease_raw = zero_pad_subset(model, comp.get('disease_raw', comp['disease']), 'disease')
        disease = zero_pad_subset(model, comp['disease'], 'disease')
        residual = zero_pad_subset(model, comp['residual'], 'residual')
    else:
        age = comp['age']
        disease = comp['disease']
        disease_raw = comp.get('disease_raw', disease)
        residual = comp['residual']
    return {
        'age': age,
        'disease_raw': disease_raw,
        'disease': disease,
        'residual': residual,
        'total': age + disease + residual,
    }


def ordered_train_subjects(specs):
    split = json.loads(Path(specs['TrainSplit']).read_text())
    seen = []
    for item in split:
        sid = int(re.search(r'ID_(\d+)_t', Path(str(item)).stem).group(1))
        if sid not in seen:
            seen.append(sid)
    return seen

label_by_sid = LABEL_DF.sort_values('tp').groupby('sid').first()['diagnosis'].to_dict()
train_auc_rows = []
for model_name, model in MODELS.items():
    subject_ids = ordered_train_subjects(model['specs'])
    if len(subject_ids) != model['latent_weights'].shape[0]:
        raise RuntimeError(f"Latent/subject count mismatch for {model['label']}: {model['latent_weights'].shape[0]} vs {len(subject_ids)}")
    z_all = model['latent_weights'].to(DEVICE)
    y = np.array([int(label_by_sid[sid]) for sid in subject_ids])
    # Raw probe: force disease condition to 1 so the raw branch capacity is not trivially gated by the true label.
    raw_components = component_vectors(model, z_all, time_value=0.5, diagnosis=1.0)
    # Oracle-gated probe: use the true label; useful for branch behavior but not a label-free test.
    diag_tensor = torch.tensor(y, dtype=z_all.dtype, device=DEVICE).view(-1, 1)
    t = torch.full((z_all.shape[0], 1), 0.5, device=DEVICE, dtype=z_all.dtype)
    with torch.no_grad():
        comp_oracle = model['flow'].velocity_components(z_all, t, t, age_cond=diag_tensor)
    if model['kind'] in ('subset128', 'subset160'):
        oracle_disease = zero_pad_subset(model, comp_oracle['disease'], 'disease')
    else:
        oracle_disease = comp_oracle['disease']
    for component in ['age', 'disease_raw', 'residual', 'total']:
        speed = raw_components[component].norm(dim=1).detach().cpu().numpy()
        train_auc_rows.append({
            'model': model['label'],
            'component': component,
            'auc': auc_score(y, speed),
            'mean_healthy_speed': float(np.mean(speed[y == 0])),
            'mean_diseased_speed': float(np.mean(speed[y == 1])),
            'probe_type': 'raw_no_true_label_gate',
        })
    speed = oracle_disease.norm(dim=1).detach().cpu().numpy()
    train_auc_rows.append({
        'model': model['label'],
        'component': 'disease_oracle_gated',
        'auc': auc_score(y, speed),
        'mean_healthy_speed': float(np.mean(speed[y == 0])),
        'mean_diseased_speed': float(np.mean(speed[y == 1])),
        'probe_type': 'oracle_gated_not_label_free',
    })

TRAIN_COMPONENT_AUC = pd.DataFrame(train_auc_rows)
TRAIN_COMPONENT_AUC.to_csv(OUTPUT_DIR / 'train_component_velocity_auc.csv', index=False)

fig = go.Figure()
for model, sub in TRAIN_COMPONENT_AUC.groupby('model'):
    fig.add_trace(go.Bar(
        x=sub['component'],
        y=sub['auc'],
        name=model,
        marker_color=MODEL_COLORS.get(model, None),
    ))
fig.add_hline(y=0.5, line_dash='dash', line_color='black')
fig.update_layout(
    title='Train-subject component-speed diagnosis AUC',
    xaxis_title='velocity component',
    yaxis_title='AUC for diagnosis from component speed',
    barmode='group',
    width=1250,
    height=560,
    margin=dict(b=130),
)
fig.update_yaxes(range=[0, 1.05])
save_plotly_figure(
    fig,
    '06_train_component_velocity_auc',
    'Train-subject component-speed diagnosis AUC',
    'Velocity disentanglement',
    'Raw components test leakage/capacity without using the true disease gate. Oracle-gated disease is shown separately and is not label-free.',
)


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/06_train_component_velocity_auc.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/06_train_component_velocity_auc.html')

In [7]:
# Shape-space utility functions for new age-60/70/80 surface plots.
def age_to_time_fit():
    fit = LABEL_DF[['age', 'age_norm']].dropna()
    slope, intercept = np.polyfit(fit['age'].to_numpy(), fit['age_norm'].to_numpy(), 1)
    return float(slope), float(intercept)

AGE_TO_TIME_SLOPE, AGE_TO_TIME_INTERCEPT = age_to_time_fit()

def age_to_time(age):
    return AGE_TO_TIME_SLOPE * float(age) + AGE_TO_TIME_INTERCEPT


def regular_grid(resolution=GRID_RES):
    axis = torch.linspace(-1.0, 1.0, int(resolution), dtype=torch.float32)
    xyz = torch.stack(torch.meshgrid(axis, axis, axis, indexing='ij'), dim=-1).reshape(-1, 3)
    return axis.numpy(), xyz


def decode_points(decoder, z, xyz, batch_size=GRID_BATCH):
    outputs = []
    with torch.no_grad():
        for start in range(0, xyz.shape[0], batch_size):
            x = xyz[start:start + batch_size].to(DEVICE)
            zz = z.reshape(1, -1).expand(x.shape[0], -1)
            outputs.append(decoder(torch.cat([zz, x], dim=1)).squeeze(-1).cpu())
    return torch.cat(outputs).numpy()


def decode_grid(decoder, z, resolution=GRID_RES):
    axis, xyz = regular_grid(resolution)
    values = decode_points(decoder, z, xyz).reshape(resolution, resolution, resolution)
    return axis, values


def mesh_from_grid(sdf_grid):
    spacing = 2.0 / (sdf_grid.shape[0] - 1)
    vertices, faces, _, _ = marching_cubes(sdf_grid, level=0.0, spacing=(spacing, spacing, spacing))
    mesh = trimesh.Trimesh(vertices=vertices - 1.0, faces=faces, process=False)
    if mesh.is_watertight and mesh.volume < 0.0:
        mesh.invert()
    return mesh


def surface_geometry(decoder, z, resolution=GRID_RES):
    _, sdf_grid = decode_grid(decoder, z, resolution)
    mesh = mesh_from_grid(sdf_grid)
    vertices = torch.as_tensor(mesh.vertices, dtype=z.dtype, device=DEVICE)
    sdf_parts, grad_parts = [], []
    for start in range(0, vertices.shape[0], SURFACE_BATCH):
        x = vertices[start:start + SURFACE_BATCH].detach().clone().requires_grad_(True)
        zz = z.detach().expand(x.shape[0], -1)
        sdf = decoder(torch.cat([zz, x], dim=1)).squeeze(-1)
        grad_x = torch.autograd.grad(sdf.sum(), x, create_graph=False)[0]
        sdf_parts.append(sdf.detach().cpu())
        grad_parts.append(grad_x.detach().cpu())
    sdf = torch.cat(sdf_parts).numpy()
    grad_x = torch.cat(grad_parts).numpy()
    grad_norm = np.linalg.norm(grad_x, axis=1) + 1e-12
    grad_unit = grad_x / grad_norm[:, None]
    mesh_normals = np.asarray(mesh.vertex_normals)
    alignment = np.einsum('ij,ij->i', grad_unit, mesh_normals)
    sign = 1.0 if np.nanmedian(alignment) >= 0.0 else -1.0
    return {
        'sdf_grid': sdf_grid,
        'mesh': mesh,
        'vertices': vertices,
        'base_sdf': sdf,
        'grad_norm': grad_norm,
        'outward_normals': sign * grad_unit,
        'sdf_to_outward_sign': sign,
        'normal_alignment': float(np.nanmedian(np.abs(alignment))),
    }


def latent_jvp_at_vertices(decoder, z, velocity, vertices):
    parts = []
    for start in range(0, vertices.shape[0], SURFACE_BATCH):
        x = vertices[start:start + SURFACE_BATCH].detach()
        z0 = z.detach().clone().requires_grad_(True)
        v0 = velocity.detach()
        def decode_latent(latent):
            zz = latent.expand(x.shape[0], -1)
            return decoder(torch.cat([zz, x], dim=1)).squeeze(-1)
        _, jvp = torch.autograd.functional.jvp(decode_latent, z0, v0, create_graph=False, strict=False)
        parts.append(jvp.detach().cpu())
    return torch.cat(parts).numpy()


def latent_jacobian_norm(decoder, z, vertices, surface_batch=512):
    norms = []
    for start in range(0, vertices.shape[0], surface_batch):
        x = vertices[start:start + surface_batch].detach()
        zz = z.detach().expand(x.shape[0], -1).clone().requires_grad_(True)
        sdf = decoder(torch.cat([zz, x], dim=1)).squeeze(-1)
        grad_z = torch.autograd.grad(sdf.sum(), zz, create_graph=False, retain_graph=False)[0]
        norms.append(torch.linalg.norm(grad_z, dim=1).detach().cpu())
    return torch.cat(norms).numpy()


def vertex_area_weights(mesh):
    weights = np.zeros(len(mesh.vertices), dtype=np.float64)
    contribution = np.asarray(mesh.area_faces, dtype=np.float64) / 3.0
    for corner in range(3):
        np.add.at(weights, np.asarray(mesh.faces)[:, corner], contribution)
    return weights


def surface_component_map(model, geometry, z, velocity, delta_t=DELTA_T):
    decoder = model['decoder']
    vertices = geometry['vertices']
    jvp = latent_jvp_at_vertices(decoder, z, velocity, vertices)
    xyz_cpu = vertices.detach().cpu()
    sdf_step = decode_points(decoder, z + delta_t * velocity, xyz_cpu)
    sign = geometry['sdf_to_outward_sign']
    normal_speed = -sign * jvp / geometry['grad_norm']
    inr_difference = geometry['base_sdf'] - sdf_step
    weights = vertex_area_weights(geometry['mesh'])
    return {
        'normal_speed': normal_speed,
        'inr_difference': inr_difference,
        'weights': weights,
        'latent_speed': float(velocity.norm().item()),
        'rms_normal_speed': float(np.sqrt(np.average(normal_speed**2, weights=weights))),
        'mean_abs_normal_speed': float(np.average(np.abs(normal_speed), weights=weights)),
        'net_volume_rate': float(np.sum(normal_speed * weights)),
        'outward_area_fraction': float(weights[normal_speed > 0].sum() / weights.sum()),
    }


def transport_state(model, z0, source_time, target_time, diagnosis=1.0):
    s = torch.full((z0.shape[0], 1), float(source_time), device=z0.device, dtype=z0.dtype)
    t = torch.full((z0.shape[0], 1), float(target_time), device=z0.device, dtype=z0.dtype)
    cond = torch.full((z0.shape[0], 1), float(diagnosis), device=z0.device, dtype=z0.dtype)
    with torch.no_grad():
        return z0 + (t - s) * model['flow'](z0, s, t, age_cond=cond)

print(f'age_norm = {AGE_TO_TIME_SLOPE:.6f} * age + {AGE_TO_TIME_INTERCEPT:.6f}')


age_norm = 0.025000 * age + -1.250000


In [8]:
# Compute age-specific mean-anchor disease surface speed, INR difference, and Jacobian sensitivity.
AGE_RESULTS = {}
age_metric_rows = []
source_time = age_to_time(REFERENCE_AGE)
for model_name, model in MODELS.items():
    AGE_RESULTS[model_name] = {}
    z_ref = model['mean_z']
    for age in AGES:
        print('Computing age surface:', model['label'], 'age', age)
        target_time = age_to_time(age)
        z_age = transport_state(model, z_ref, source_time, target_time, diagnosis=EVAL_DIAGNOSIS)
        velocities = component_vectors(model, z_age, time_value=target_time, diagnosis=EVAL_DIAGNOSIS)
        geometry = surface_geometry(model['decoder'], z_age, GRID_RES)
        disease_map = surface_component_map(model, geometry, z_age, velocities['disease'])
        total_map = surface_component_map(model, geometry, z_age, velocities['total'])
        age_map = surface_component_map(model, geometry, z_age, velocities['age'])
        residual_map = surface_component_map(model, geometry, z_age, velocities['residual'])
        jac_norm = latent_jacobian_norm(model['decoder'], z_age, geometry['vertices'])
        AGE_RESULTS[model_name][age] = {
            'z': z_age,
            'time': target_time,
            'geometry': geometry,
            'velocities': velocities,
            'disease': disease_map,
            'total': total_map,
            'age_component': age_map,
            'residual': residual_map,
            'jacobian_norm': jac_norm,
        }
        age_metric_rows.append({
            'model_key': model_name,
            'model': model['label'],
            'age': age,
            'disease_latent_speed': disease_map['latent_speed'],
            'disease_shape_rms_speed': disease_map['rms_normal_speed'],
            'disease_net_volume_rate': disease_map['net_volume_rate'],
            'total_shape_rms_speed': total_map['rms_normal_speed'],
            'age_shape_rms_speed': age_map['rms_normal_speed'],
            'residual_shape_rms_speed': residual_map['rms_normal_speed'],
            'mean_jacobian_norm': float(np.mean(jac_norm)),
            'p95_jacobian_norm': float(np.quantile(jac_norm, 0.95)),
        })

AGE_SPEED_METRICS = pd.DataFrame(age_metric_rows)
AGE_SPEED_METRICS.to_csv(OUTPUT_DIR / 'age_60_70_80_best3_speed_metrics.csv', index=False)
print('Saved age speed metrics CSV for reproducibility, but report uses plots only.')


Computing age surface: Subset 64/128/64 best age 60.0
Computing age surface: Subset 64/128/64 best age 70.0
Computing age surface: Subset 64/128/64 best age 80.0
Computing age surface: Subset 64/160/32 E2 best saved age 60.0
Computing age surface: Subset 64/160/32 E2 best saved age 70.0
Computing age surface: Subset 64/160/32 E2 best saved age 80.0
Computing age surface: Additive strong best age 60.0
Computing age surface: Additive strong best age 70.0
Computing age surface: Additive strong best age 80.0
Saved age speed metrics CSV for reproducibility, but report uses plots only.


In [9]:
# Plot latent speed versus shape-space speed and compare model speed trends with real ground-truth trends.
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Disease latent speed ||v_d||',
        'Disease shape RMS normal speed',
        'Disease net volume-rate direction',
        'Decoder Jacobian sensitivity',
    ),
)
for model, sub in AGE_SPEED_METRICS.groupby('model'):
    sub = sub.sort_values('age')
    color = MODEL_COLORS.get(model, None)
    fig.add_trace(go.Scatter(x=sub['age'], y=sub['disease_latent_speed'], mode='lines+markers', name=model, line=dict(color=color)), row=1, col=1)
    fig.add_trace(go.Scatter(x=sub['age'], y=sub['disease_shape_rms_speed'], mode='lines+markers', name=model, line=dict(color=color), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=sub['age'], y=sub['disease_net_volume_rate'], mode='lines+markers', name=model, line=dict(color=color), showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=sub['age'], y=sub['mean_jacobian_norm'], mode='lines+markers', name=model, line=dict(color=color), showlegend=False), row=2, col=2)
fig.add_hline(y=0.0, line_color='black', line_width=1, row=2, col=1)
fig.update_layout(
    title='Age-specific latent speed vs decoded shape-space speed at ages 60, 70, 80',
    width=1250,
    height=760,
)
fig.update_xaxes(title='age')
save_plotly_figure(
    fig,
    '07_age_specific_latent_vs_shape_speed',
    'Age-specific latent speed versus shape-space speed',
    'Age 60/70/80 speed',
    'Latent speed is not anatomical by itself; decoded RMS normal speed is the shape-space effect.',
)

# Normalized comparison against real speed trends. Units are different, so this compares age trend direction only.
def normalize_abs(series):
    arr = np.asarray(series, dtype=float)
    scale = np.nanmax(np.abs(arr))
    if not np.isfinite(scale) or scale < 1e-12:
        return arr * 0.0
    return arr / scale

real_disease = REAL_AGE_SUMMARY[REAL_AGE_SUMMARY['diagnosis'] == 1].sort_values('nearest_age')
real_healthy = REAL_AGE_SUMMARY[REAL_AGE_SUMMARY['diagnosis'] == 0].sort_values('nearest_age')
fig = make_subplots(rows=1, cols=2, subplot_titles=('Aging/thickness trend', 'Disease/bump trend'))
fig.add_trace(go.Scatter(
    x=real_healthy['nearest_age'], y=normalize_abs(real_healthy['thickness_rate']),
    mode='lines+markers', name='GT healthy thickness speed', line=dict(color='#147D92', dash='dash'),
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=real_disease['nearest_age'], y=normalize_abs(real_disease['thickness_rate']),
    mode='lines+markers', name='GT diseased thickness speed', line=dict(color='#D1495B', dash='dash'),
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=real_disease['nearest_age'], y=normalize_abs(real_disease['bump_rate']),
    mode='lines+markers', name='GT diseased bump speed', line=dict(color='black', dash='dash'),
), row=1, col=2)
for model, sub in AGE_SPEED_METRICS.groupby('model'):
    sub = sub.sort_values('age')
    color = MODEL_COLORS.get(model, None)
    fig.add_trace(go.Scatter(
        x=sub['age'], y=normalize_abs(sub['age_shape_rms_speed']),
        mode='lines+markers', name=f'{SHORT_NAMES.get(model, model)} model age speed',
        line=dict(color=color),
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=sub['age'], y=normalize_abs(sub['disease_shape_rms_speed']),
        mode='lines+markers', name=f'{SHORT_NAMES.get(model, model)} model disease speed',
        line=dict(color=color),
    ), row=1, col=2)
fig.update_layout(
    title='Normalized real-speed trends versus model-inferred shape speed trends',
    width=1250,
    height=520,
)
fig.update_xaxes(title='age')
fig.update_yaxes(title='normalized trend, unitless')
save_plotly_figure(
    fig,
    '08_normalized_real_vs_model_speed_trends',
    'Normalized real speed versus model-inferred shape speed trends',
    'Ground-truth speed',
    'Model speed and metadata speed have different units; this plot compares trend direction and relative age dependence.',
)


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/07_age_specific_latent_vs_shape_speed.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/08_normalized_real_vs_model_speed_trends.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/08_normalized_real_vs_model_speed_trends.html')

In [10]:
# Surface color-map figures: disease normal speed, INR finite-step difference, and decoder Jacobian sensitivity.
def robust_symmetric_limit(arrays, quantile=0.98):
    vals = np.concatenate([np.ravel(np.asarray(a)[np.isfinite(a)]) for a in arrays if np.size(a)])
    if vals.size == 0:
        return 1.0
    limit = float(np.quantile(np.abs(vals), quantile))
    return limit if limit > 0 else 1.0


def robust_positive_limit(arrays, quantile=0.98):
    vals = np.concatenate([np.ravel(np.asarray(a)[np.isfinite(a)]) for a in arrays if np.size(a)])
    if vals.size == 0:
        return 1.0
    limit = float(np.quantile(vals, quantile))
    return limit if limit > 0 else 1.0


def mesh_surface_trace(mesh, values, cmin, cmax, colorscale='RdBu_r', show_colorbar=False, title=''):
    vertices = np.asarray(mesh.vertices)
    faces = np.asarray(mesh.faces)
    return go.Mesh3d(
        x=vertices[:, 0], y=vertices[:, 1], z=vertices[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        intensity=np.asarray(values),
        colorscale=colorscale,
        cmin=cmin,
        cmax=cmax,
        showscale=show_colorbar,
        colorbar=dict(title=title, thickness=18, len=0.72),
        opacity=1.0,
        flatshading=False,
        lighting=dict(ambient=0.58, diffuse=0.72, specular=0.18, roughness=0.55),
        hovertemplate='value=%{intensity:.5f}<extra></extra>',
    )


def age_surface_figure(value_getter, title, colorscale='RdBu_r', signed=True, colorbar_title='value'):
    model_names = list(MODELS.keys())
    arrays = [value_getter(model_name, age) for model_name in model_names for age in AGES]
    if signed:
        limit = robust_symmetric_limit(arrays)
        cmin, cmax = -limit, limit
    else:
        cmin, cmax = 0.0, robust_positive_limit(arrays)
    fig = make_subplots(
        rows=len(model_names), cols=len(AGES),
        specs=[[{'type': 'scene'}] * len(AGES) for _ in model_names],
        row_titles=[MODELS[name]['label'] for name in model_names],
        column_titles=[f'Age {age:.0f}' for age in AGES],
        horizontal_spacing=0.012,
        vertical_spacing=0.015,
    )
    for row, model_name in enumerate(model_names, start=1):
        for col, age in enumerate(AGES, start=1):
            item = AGE_RESULTS[model_name][age]
            fig.add_trace(
                mesh_surface_trace(
                    item['geometry']['mesh'],
                    value_getter(model_name, age),
                    cmin,
                    cmax,
                    colorscale=colorscale,
                    show_colorbar=(row == 1 and col == len(AGES)),
                    title=colorbar_title,
                ),
                row=row, col=col,
            )
    for scene_name in [key for key in fig.layout if str(key).startswith('scene')]:
        fig.layout[scene_name].update(
            aspectmode='data',
            xaxis_visible=False,
            yaxis_visible=False,
            zaxis_visible=False,
            camera=dict(eye=dict(x=1.55, y=1.55, z=1.05)),
        )
    fig.update_layout(
        title=f'{title}<br><sup>Rows are models, columns are ages 60/70/80. Shared color scale across all panels.</sup>',
        width=360 * len(AGES),
        height=330 * len(model_names),
        margin=dict(l=150, r=35, t=95, b=20),
    )
    return fig

fig = age_surface_figure(
    lambda model_name, age: AGE_RESULTS[model_name][age]['disease']['normal_speed'],
    'Disease component signed surface-normal speed',
    colorscale='RdBu_r',
    signed=True,
    colorbar_title='normal speed',
)
save_plotly_figure(
    fig,
    '09_age_60_70_80_disease_surface_speed_maps',
    'Age 60/70/80 disease surface speed maps',
    'Surface maps',
    'Red means outward disease-component motion; blue means inward disease-component motion.',
)

fig = age_surface_figure(
    lambda model_name, age: AGE_RESULTS[model_name][age]['disease']['inr_difference'],
    'Disease component INR(z) - INR(z + v * DELTA_T)',
    colorscale='RdBu_r',
    signed=True,
    colorbar_title='INR diff',
)
save_plotly_figure(
    fig,
    '10_age_60_70_80_disease_inr_difference_maps',
    'Age 60/70/80 disease INR difference maps',
    'Surface maps',
    'Finite-step field-change view requested by the professor: INR(z) - INR(z + v * DELTA_T).',
)

fig = age_surface_figure(
    lambda model_name, age: AGE_RESULTS[model_name][age]['jacobian_norm'],
    'Decoder latent-Jacobian sensitivity on decoded age surfaces',
    colorscale='Viridis',
    signed=False,
    colorbar_title='||dINR/dz||',
)
save_plotly_figure(
    fig,
    '11_age_60_70_80_decoder_jacobian_sensitivity_maps',
    'Age 60/70/80 decoder Jacobian sensitivity maps',
    'Jacobian sensitivity',
    'High values show where the decoder strongly converts latent perturbation into INR/shape change.',
)


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/09_age_60_70_80_disease_surface_speed_maps.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/10_age_60_70_80_disease_inr_difference_maps.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/11_age_60_70_80_decoder_jacobian_sensitivity_maps.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/11_age_60_70_80_decoder_jacobian_sensitivity_maps.html')

In [11]:
# Final model-selection plots and written conclusion.
# Convert several criteria into simple 0-1 visual scores. These are for decision support, not a formal benchmark.
# Recompute the compact held-out summary here so this cell is robust if earlier plotting cells were skipped.
held_for_score = HELDOUT_SUMMARY.copy()
held_model_for_score = held_for_score.groupby('model').agg(
    mean_age=('age', 'mean'),
    mean_disease=('disease', 'mean'),
    mean_residual=('residual', 'mean'),
    mean_total=('total', 'mean'),
    mean_residual_fraction=('residual_fraction', 'mean'),
).reset_index()
healthy_for_score = held_for_score[held_for_score['true_diagnosis'] == 0][['model', 'disease']].rename(columns={'disease': 'healthy_false_disease_speed'})
diseased_for_score = held_for_score[held_for_score['true_diagnosis'] == 1][['model', 'disease']].rename(columns={'disease': 'diseased_disease_speed'})
held_model_for_score = held_model_for_score.merge(healthy_for_score, on='model', how='left')
held_model_for_score = held_model_for_score.merge(diseased_for_score, on='model', how='left')
score_rows = []
held_avg = held_model_for_score.set_index('model')
ckpt = CHECKPOINT.set_index('model')
selected_diag = DIAGNOSIS_METRICS[DIAGNOSIS_METRICS['selected_for_notebook'].astype(bool)]
selected_auc = float(selected_diag['auc'].iloc[0]) if len(selected_diag) else np.nan
selected_bal = float(selected_diag['balanced_accuracy'].iloc[0]) if len(selected_diag) else np.nan

# Helper normalizations.
def inverse_minmax(values):
    arr = np.asarray(values, dtype=float)
    lo, hi = np.nanmin(arr), np.nanmax(arr)
    if hi - lo < 1e-12:
        return np.ones_like(arr)
    return (hi - arr) / (hi - lo)

def minmax(values):
    arr = np.asarray(values, dtype=float)
    lo, hi = np.nanmin(arr), np.nanmax(arr)
    if hi - lo < 1e-12:
        return np.ones_like(arr)
    return (arr - lo) / (hi - lo)

models = list(ckpt.index)
resid_score = inverse_minmax([held_avg.loc[m, 'mean_residual_fraction'] for m in models])
disease_sep_score = minmax([
    held_avg.loc[m, 'diseased_disease_speed'] - held_avg.loc[m, 'healthy_false_disease_speed']
    for m in models
])
forecast_score = inverse_minmax([ckpt.loc[m, 'reported_or_current_test_chamfer'] for m in models])
for i, model in enumerate(models):
    has_label_free = model == 'Subset 64/160/32 E2 best saved'
    score_rows.extend([
        {'model': model, 'criterion': 'low residual leakage', 'score': float(resid_score[i])},
        {'model': model, 'criterion': 'disease separation', 'score': float(disease_sep_score[i])},
        {'model': model, 'criterion': 'reported forecast Chamfer', 'score': float(forecast_score[i])},
        {'model': model, 'criterion': 'label-free disease head', 'score': float(max(selected_auc, selected_bal)) if has_label_free else 0.0},
    ])
SCORE_DF = pd.DataFrame(score_rows)
SCORE_DF.to_csv(OUTPUT_DIR / 'model_selection_scores.csv', index=False)

fig = go.Figure()
for model, sub in SCORE_DF.groupby('model'):
    fig.add_trace(go.Bar(
        x=sub['criterion'],
        y=sub['score'],
        name=model,
        marker_color=MODEL_COLORS.get(model, None),
    ))
fig.update_layout(
    title='Decision-support scores for the three selected models',
    xaxis_title='criterion',
    yaxis_title='normalized score, higher is better',
    barmode='group',
    width=1250,
    height=560,
    margin=dict(b=130),
)
fig.update_yaxes(range=[0, 1.05])
save_plotly_figure(
    fig,
    '12_model_selection_scores',
    'Model-selection score plot',
    'Final decision',
    'Scores summarize the current evidence. They are for interpretation, not a formal statistical benchmark.',
)

save_text_page(
    '13_final_recommendation',
    'Final recommendation from current outputs',
    'Final decision',
    """
    <h2>Best velocity disentanglement right now</h2>
    <p><strong>Additive strong best</strong> is the cleanest supervised/oracle disentanglement baseline. It has high disease speed on the diseased held-out subject, zero disease speed on the healthy held-out subject because the disease condition is oracle-gated, and low residual leakage.</p>

    <h2>Best architecture direction for the paper target</h2>
    <p><strong>Subset 64/160/32</strong> is the most aligned with the desired test-time formulation because it has a disease-prediction head and extremely low residual shape speed. However, the selected saved checkpoint predicts the healthy held-out subject as diseased, so label-free disease inference is currently not reliable enough.</p>

    <h2>Best future/forecast checkpoint among saved models</h2>
    <p><strong>Subset 64/128/64</strong> has the best reported saved-checkpoint Chamfer, but its velocity disentanglement is weak because residual velocity dominates shape change.</p>

    <h2>Important failure to fix next</h2>
    <p>The <code>64/160/32</code> model is promising but needs better disease-head calibration and validation. Its residual branch is clean, but false-positive disease prediction creates false disease speed on a healthy subject. The next experiment should keep the 64/160/32 block design and improve disease prediction/calibration rather than returning to the residual-heavy 64/128/64 design.</p>

    <h2>How to read the surface colors</h2>
    <p>Surface colors are not diagnosis probability. They are signed anatomical motion induced by a velocity component through the decoder Jacobian. Red/blue indicate opposite normal directions. INR-difference maps show finite field change; Jacobian maps show decoder sensitivity.</p>
    """,
    'Text conclusion summarizing disentanglement, disease prediction, future generation, and the main next fix.',
)


Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/12_model_selection_scores.html
Saved output: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/13_final_recommendation.html


PosixPath('/home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/13_final_recommendation.html')

In [12]:
# Refresh the report index after running the cells above.
index_path = write_figure_index()
try:
    display_path = index_path.relative_to(ROOT).as_posix()
except ValueError:
    display_path = str(index_path)
display(HTML(
    f"<p><strong>All model-selection outputs:</strong> "
    f"<a href='{display_path}' target='_blank'>{display_path}</a></p>"
))
print('Saved analysis:', OUTPUT_DIR)
print('Open this file for every saved explanation and figure:', index_path)


Figure index: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/index.html


Saved analysis: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed
Open this file for every saved explanation and figure: /home/jakaria/INR/Deep3DComp/analysis_torus_best3_disentanglement_future_speed/figures/index.html
